# 3.4 softmax 回归

前面三节讨论线性回归：输入若干特征，预测一个连续数值。本节转向**分类问题**：模型不再回答“多少”，而是回答“属于哪一类”，并给出属于各类别的概率。

softmax 回归虽然名称中含有“回归”，但它是解决多分类问题的线性模型，也是后续图像分类和深度神经网络分类器的基础。

本 Notebook 按《动手学深度学习》原版 3.4 的知识顺序整理，并使用少量 PyTorch 代码验证公式。

对应教材：[3.4 softmax 回归](https://zh.d2l.ai/chapter_linear-networks/softmax-regression.html)

In [ ]:
import torch  # 导入 PyTorch，用于张量计算和自动微分
from torch.nn import functional as F  # 导入常用神经网络函数，并简写为 F

torch.manual_seed(42)  # 固定随机种子，使演示结果可复现
print("PyTorch version:", torch.__version__)  # 输出当前 PyTorch 版本

PyTorch version: 2.13.0+cpu


## 3.4.1 分类问题

回归预测连续值，例如房价、住院天数；分类预测离散类别，例如邮件是否为垃圾邮件、图像是猫还是狗。

分类输出可以有两种理解：

- **硬分类**：只输出最可能的类别；
- **软分类**：输出样本属于每个类别的概率。

即使最终只关心硬类别，训练时通常仍使用软概率。

假设输入是 $2\times2$ 灰度图像。展开后得到 4 个特征 $x_1,x_2,x_3,x_4$，类别为“猫”“鸡”“狗”。普通类别之间没有自然大小顺序，因此不应把类别编号 0、1、2 当作连续数值进行回归。

### 独热编码

独热编码（one-hot encoding）使用长度等于类别数的向量表示标签：真实类别位置为 1，其余位置为 0。

$$
\mathbf y\in\{(1,0,0),(0,1,0),(0,0,1)\}.\tag{3.4.1}
$$

| 类别 | 类别索引 | 独热标签 |
|---|---:|---|
| 猫 | 0 | $(1,0,0)$ |
| 鸡 | 1 | $(0,1,0)$ |
| 狗 | 2 | $(0,0,1)$ |

类别索引只是标识，不表示“狗大于鸡”或类别间距离。

In [2]:
class_names = ["猫", "鸡", "狗"]  # 使用 Python 列表保存类别名称，索引对应类别编号
label_indices = torch.tensor([0, 1, 2, 1])  # 四个样本的类别索引分别为猫、鸡、狗、鸡
one_hot_labels = F.one_hot(label_indices, num_classes=3)  # 把类别索引转换为长度为 3 的独热向量

print("类别索引:", label_indices)  # 输出紧凑的整数标签
print("独热标签:\n", one_hot_labels)  # 输出每个标签对应的独热编码
print("第一个类别名称:", class_names[label_indices[0].item()])  # 用整数索引从列表中取类别名称

类别索引: tensor([0, 1, 2, 1])
独热标签:
 tensor([[1, 0, 0],
        [0, 1, 0],
        [0, 0, 1],
        [0, 1, 0]])
第一个类别名称: 猫


### Python 语法补充

- `class_names = ["猫", "鸡", "狗"]`：方括号创建列表，元素按顺序保存。
- `class_names[0]`：索引从 0 开始，取列表第一个元素。
- `F.one_hot(..., num_classes=3)`：关键字参数明确指定类别总数。
- 字符串中的 `\n` 表示换行。

## 3.4.2 网络架构

为了估计每个类别的条件概率，模型必须为每个类别产生一个输出。4 个输入特征、3 个类别需要 3 个仿射函数：

$$
\begin{aligned}
o_1 &= x_1w_{11}+x_2w_{12}+x_3w_{13}+x_4w_{14}+b_1,\\
o_2 &= x_1w_{21}+x_2w_{22}+x_3w_{23}+x_4w_{24}+b_2,\\
o_3 &= x_1w_{31}+x_2w_{32}+x_3w_{33}+x_4w_{34}+b_3.
\end{aligned}\tag{3.4.2}
$$

$o_1,o_2,o_3$ 称为 **logit**，即尚未规范化的预测分数。logit 可以为负，也不要求总和为 1，因此还不是概率。

![softmax 回归是一种单层神经网络](https://zh.d2l.ai/_images/softmaxreg.svg)

每个输出都依赖全部输入，所以输出层是全连接层。softmax 回归与线性回归一样，都是单层神经网络。

向量形式为

$$
\mathbf o=\mathbf W\mathbf x+\mathbf b.
$$

教材按单样本写法令 $\mathbf W\in\mathbb R^{3\times4}$。PyTorch 的 `nn.Linear(4, 3)` 也按 `[输出数, 输入数]` 保存权重。

In [3]:
x = torch.tensor([0.2, 0.5, 0.1, 0.7])  # 创建一个包含 4 个特征的样本
W = torch.tensor([[0.1, 0.2, -0.1, 0.3],  # 第 1 行是“猫”类别的 4 个权重
                  [-0.2, 0.1, 0.4, 0.2],  # 第 2 行是“鸡”类别的 4 个权重
                  [0.3, -0.1, 0.2, -0.2]])  # 第 3 行是“狗”类别的 4 个权重
b = torch.tensor([0.0, 0.1, -0.1])  # 每个类别拥有一个偏置
logits = W @ x + b  # 计算三个类别的未规范化分数

print("x shape:", x.shape)  # 单个样本包含 4 个特征
print("W shape:", W.shape)  # 权重形状为 3 个输出乘 4 个输入
print("logits:", logits)  # 输出三个类别分数
print("logits shape:", logits.shape)  # 结果包含 3 个类别分数

x shape: torch.Size([4])
W shape: torch.Size([3, 4])
logits: tensor([ 0.3200,  0.2900, -0.2100])
logits shape: torch.Size([3])


## 3.4.3 全连接层的参数开销

具有 $d$ 个输入、$q$ 个输出的全连接层包含：

- 权重：$dq$ 个；
- 偏置：$q$ 个；
- 总参数：$dq+q$ 个；
- 主要参数和计算开销：$\mathcal O(dq)$。

本例 $d=4,q=3$，共有 $4\times3+3=15$ 个参数。输入和输出维度很大时，全连接层成本会迅速增加。后续模型会使用低秩分解、稀疏连接或其他结构降低开销。

In [4]:
def linear_parameter_count(input_count, output_count, use_bias=True):  # 定义计算全连接层参数量的函数
    weight_count = input_count * output_count  # 每个输出与每个输入相连，因此权重数为 d×q
    bias_count = output_count if use_bias else 0  # 条件表达式：使用偏置时有 q 个，否则为 0
    return weight_count + bias_count  # 返回权重和偏置的总数量

print("4 输入、3 输出的参数量:", linear_parameter_count(4, 3))  # 输出 15
print("784 输入、10 输出的参数量:", linear_parameter_count(784, 10))  # Fashion-MNIST 线性分类器有 7850 个参数

4 输入、3 输出的参数量: 15
784 输入、10 输出的参数量: 7850


## 3.4.4 softmax 运算

logit 不能直接解释为概率，因为它可能为负，且总和不一定为 1。softmax 将 logit 转换为合法概率分布：

$$
\hat{\mathbf y}=\operatorname{softmax}(\mathbf o),\qquad
\hat y_j=\frac{\exp(o_j)}{\sum_k\exp(o_k)}.\tag{3.4.3}
$$

softmax 具有三个关键性质：

1. 指数函数保证每项为正；
2. 除以所有指数值之和，保证概率总和为 1；
3. 指数函数单调，因此不改变各 logit 的大小顺序。

所以预测类别可以直接从 logit 中选择：

$$
\operatorname*{argmax}_j\hat y_j
=\operatorname*{argmax}_jo_j.\tag{3.4.4}
$$

例如概率为 $(0.1,0.8,0.1)$ 时，模型选择第 2 个类别“鸡”。

### 校准

概率模型不仅应分类正确，还应尽量校准。例如模型对大量样本都给出 0.8 置信度时，理想情况下其中约 80% 应预测正确。

虽然 softmax 是非线性运算，但 logit 仍是输入的仿射函数，分类边界仍是线性的，因此 softmax 回归属于线性模型。

In [5]:
def stable_softmax(logits):  # 定义数值稳定的 softmax 函数
    shifted_logits = logits - logits.max(dim=-1, keepdim=True).values  # 每行减去最大值，避免 exp 上溢
    exp_values = torch.exp(shifted_logits)  # 对平移后的每个 logit 求指数，保证结果为正
    return exp_values / exp_values.sum(dim=-1, keepdim=True)  # 每项除以本行总和，使概率和为 1

probabilities = stable_softmax(logits)  # 把前面三个 logit 转换成概率
predicted_from_logits = logits.argmax()  # 直接取最大 logit 的索引
predicted_from_probabilities = probabilities.argmax()  # 取最大概率的索引

print("logits:", logits)  # 输出未规范化分数
print("probabilities:", probabilities)  # 输出三个类别概率
print("概率和:", probabilities.sum().item())  # 概率和应接近 1
print("两个 argmax 相同:", predicted_from_logits.item() == predicted_from_probabilities.item())  # 验证 softmax 不改变顺序

logits: tensor([ 0.3200,  0.2900, -0.2100])
probabilities: tensor([0.3908, 0.3792, 0.2300])
概率和: 1.0
两个 argmax 相同: True


### Python 与 PyTorch 语法补充

- `dim=-1`：沿最后一维操作；对类别分数而言，最后一维通常就是类别维。
- `keepdim=True`：求最大值或求和后保留长度为 1 的维度，便于后续广播。
- `.values`：`torch.max` 同时返回最大值和索引，这里只取最大值。
- `a == b`：比较两个值，结果为布尔值 `True` 或 `False`。

减去每行最大 logit 不改变 softmax：分子分母会同时乘上同一个常数 $e^{-\max(o)}$，最终抵消。

## 3.4.5 小批量样本的矢量化

逐个样本计算无法充分利用硬件。设批量大小为 $n$、输入特征数为 $d$、类别数为 $q$：

$$
\mathbf X\in\mathbb R^{n\times d},\quad
\mathbf W\in\mathbb R^{d\times q},\quad
\mathbf b\in\mathbb R^{1\times q}.
$$

整个批量一次计算：

$$
\begin{aligned}
\mathbf O&=\mathbf X\mathbf W+\mathbf b,\\
\hat{\mathbf Y}&=\operatorname{softmax}(\mathbf O).
\end{aligned}\tag{3.4.5}
$$

$\mathbf O$ 和 $\hat{\mathbf Y}$ 的形状均为 $n\times q$。偏置 $\mathbf b$ 通过广播加到每一行。softmax 必须按行执行：每个样本在自己的 $q$ 个类别之间归一化。

In [6]:
batch_X = torch.randn(5, 4)  # 创建 5 个样本，每个样本包含 4 个特征
batch_W = torch.randn(4, 3)  # 教材批量写法：4 个输入映射为 3 个类别
batch_b = torch.zeros(1, 3)  # 创建形状为 [1, 3] 的偏置，稍后广播到 5 行
batch_logits = batch_X @ batch_W + batch_b  # 一次计算整个批量的类别分数
batch_probabilities = stable_softmax(batch_logits)  # 沿每行类别维计算 softmax

print("X shape:", batch_X.shape)  # [5, 4]
print("W shape:", batch_W.shape)  # [4, 3]
print("logits shape:", batch_logits.shape)  # [5, 3]
print("每行概率和:", batch_probabilities.sum(dim=1))  # 每个样本的类别概率和都应为 1

X shape: torch.Size([5, 4])
W shape: torch.Size([4, 3])
logits shape: torch.Size([5, 3])
每行概率和: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


## 3.4.6 损失函数

模型已经能够输出类别概率，下一步需要定义目标函数，衡量预测概率与真实标签的差异。

### 3.4.6.1 对数似然

softmax 输出可以解释为条件概率，例如

$$
\hat y_1=P(y=\text{猫}\mid\mathbf x).
$$

若数据集含 $n$ 个相互独立的样本，整个数据集的似然为

$$
P(\mathbf Y\mid\mathbf X)
=\prod_{i=1}^nP(\mathbf y^{(i)}\mid\mathbf x^{(i)}).\tag{3.4.6}
$$

最大化似然等价于最小化负对数似然：

$$
-\log P(\mathbf Y\mid\mathbf X)
=\sum_{i=1}^n-\log P(\mathbf y^{(i)}\mid\mathbf x^{(i)})
=\sum_{i=1}^nl(\mathbf y^{(i)},\hat{\mathbf y}^{(i)}).\tag{3.4.7}
$$

单个样本的交叉熵损失为

$$
l(\mathbf y,\hat{\mathbf y})
=-\sum_{j=1}^qy_j\log\hat y_j.\tag{3.4.8}
$$

独热标签只有真实类别位置为 1，因此损失可化简为

$$
l=-\log(\text{真实类别的预测概率}).
$$

真实类别概率越接近 1，损失越接近 0；概率越接近 0，损失越大。

In [ ]:
sample_logits = torch.tensor([[2.0, 1.0, 0.1],  # 第一个样本的三个类别分数
                            [0.2, 0.3, 2.1]])  # 第二个样本的三个类别分数
sample_labels = torch.tensor([0, 2])  # 第一个样本真实类别为 0，第二个为 2
sample_probs = stable_softmax(sample_logits)  # 把两个样本的 logits 转换成概率
row_indices = torch.arange(len(sample_labels))  # 生成行索引 [0, 1]
true_class_probs = sample_probs[row_indices, sample_labels]  # 使用高级索引取出每行真实类别概率
manual_loss = -torch.log(true_class_probs)  # 按 -log(真实类别概率) 手工计算损失
builtin_loss = F.cross_entropy(sample_logits, sample_labels, reduction="none")  # PyTorch 直接从 logits 计算交叉熵

print("类别概率:\n", sample_probs)  # 输出每个样本的概率分布
print("真实类别概率:", true_class_probs)  # 输出两个正确类别的概率
print("手工交叉熵:", manual_loss)  # 输出手工计算结果
print("PyTorch 交叉熵:", builtin_loss)  # 两种结果应一致

类别概率:
 tensor([[0.6590, 0.2424, 0.0986],
        [0.1138, 0.1257, 0.7605]])
真实类别概率: tensor([0.6590, 0.7605])
手工交叉熵: tensor([0.4170, 0.2737])
PyTorch 交叉熵: tensor([0.4170, 0.2737])


### PyTorch 使用注意

`F.cross_entropy(logits, labels)` 接收**未经 softmax 的 logits**和整数类别索引。它内部组合了数值稳定的 `log_softmax` 与负对数似然。不要先手动 softmax 再传入 `cross_entropy`。

`sample_probs[row_indices, sample_labels]` 使用高级索引：第 0 行取第 `sample_labels[0]` 列，第 1 行取第 `sample_labels[1]` 列。

### 3.4.6.2 softmax 及其导数

将 softmax 代入交叉熵：

$$
\begin{aligned}
l(\mathbf y,\hat{\mathbf y})
&=-\sum_{j=1}^qy_j\log\frac{\exp(o_j)}{\sum_{k=1}^q\exp(o_k)}\\
&=\log\sum_{k=1}^q\exp(o_k)-\sum_{j=1}^qy_jo_j.
\end{aligned}\tag{3.4.9}
$$

损失关于 logit $o_j$ 的导数非常简洁：

$$
\frac{\partial l}{\partial o_j}
=\operatorname{softmax}(\mathbf o)_j-y_j
=\hat y_j-y_j.\tag{3.4.10}
$$

梯度就是“模型预测概率减去真实独热标签”。这与平方损失中“预测值减真实值”的结构相似。

- 对真实类别：$y_j=1$，梯度通常为负，梯度下降会提高该 logit；
- 对错误类别：$y_j=0$，梯度为正，梯度下降会降低该 logit。

In [8]:
gradient_logits = torch.tensor([[2.0, 1.0, 0.1]], requires_grad=True)  # 创建需要计算梯度的一行 logits
gradient_label = torch.tensor([0])  # 真实类别为第 0 类
gradient_loss = F.cross_entropy(gradient_logits, gradient_label)  # 计算单样本交叉熵
gradient_loss.backward()  # 自动计算损失关于三个 logits 的梯度

expected_gradient = stable_softmax(gradient_logits.detach()) - F.one_hot(gradient_label, 3).float()  # 按 ŷ-y 计算理论梯度
print("自动微分梯度:", gradient_logits.grad)  # 输出 autograd 得到的梯度
print("softmax - one_hot:", expected_gradient)  # 输出公式得到的梯度
print("两者一致:", torch.allclose(gradient_logits.grad, expected_gradient))  # 检查数值是否近似相等

自动微分梯度: tensor([[-0.3410,  0.2424,  0.0986]])
softmax - one_hot: tensor([[-0.3410,  0.2424,  0.0986]])
两者一致: True


### 3.4.6.3 交叉熵损失

独热标签表示确定类别，但交叉熵也适用于软标签。例如真实分布可以是 $(0.1,0.2,0.7)$。此时

$$
H(P,Q)=-\sum_jP(j)\log Q(j)
$$

衡量真实分布 $P$ 与预测分布 $Q$ 的差异。交叉熵是分类问题最常用的损失之一。

## 3.4.7 信息论基础

信息论研究如何编码、解码、传输并尽可能简洁地表示信息。它为交叉熵提供另一种解释。

### 3.4.7.1 熵

分布 $P$ 的熵定义为等等

$$
H[P]=\sum_j-P(j)\log P(j).\tag{3.4.11}
$$

熵表示按最优方式编码来自分布 $P$ 的事件时，平均至少需要多少信息。自然对数对应单位 **纳特（nat）**；$1$ nat 约等于 $1/\log2\approx1.44$ bit。

确定分布如 $(1,0,0)$ 的熵为 0，因为结果完全可预测；均匀分布如 $(1/3,1/3,1/3)$ 的熵更大，因为结果更难预测。

### 3.4.7.2 信息量

事件 $j$ 的信息量（也可理解为惊异程度）为

$$
I(j)=-\log P(j)=\log\frac{1}{P(j)}.
$$

高概率事件并不令人意外，信息量小；低概率事件更意外，信息量大。熵是事件信息量在真实分布下的期望。

### 3.4.7.3 重新审视交叉熵

交叉熵 $H(P,Q)$ 可以理解为：数据真实来自 $P$，但观察者使用主观分布 $Q$ 编码时的预期惊异。

- $P$：真实标签分布；
- $Q$：模型预测分布；
- 当 $P=Q$ 时交叉熵最低。

所以分类中的交叉熵目标有两个等价角度：

1. 最大化观测标签的似然；
2. 最小化使用模型概率编码真实标签所需的信息量。

In [9]:
def entropy(probability_distribution):  # 定义离散概率分布的熵
    positive_probs = probability_distribution[probability_distribution > 0]  # 去掉零概率，避免计算 log(0)
    return -(positive_probs * torch.log(positive_probs)).sum()  # 按 -Σp log p 计算熵

certain_distribution = torch.tensor([1.0, 0.0, 0.0])  # 完全确定的三分类分布
uniform_distribution = torch.tensor([1 / 3, 1 / 3, 1 / 3])  # 三个类别等概率分布
print("确定分布的熵:", entropy(certain_distribution).item())  # 结果应为 0
print("均匀分布的熵:", entropy(uniform_distribution).item())  # 结果应为 log(3)
print("log(3):", torch.log(torch.tensor(3.0)).item())  # 验证均匀三分类分布的熵

确定分布的熵: -0.0
均匀分布的熵: 1.0986123085021973
log(3): 1.0986123085021973


## 3.4.8 模型预测和评估

训练完成后，模型为每个样本输出 $q$ 个类别概率。通常选择概率最大的类别：

$$
\hat y=\operatorname*{argmax}_j\hat y_j.
$$

如果预测类别与真实类别相同，则该样本预测正确。准确率定义为

$$
\mathrm{accuracy}
=\frac{\text{正确预测数}}{\text{预测总数}}.
$$

准确率只判断最终类别是否正确，不评价概率是否校准，也不能体现错误预测的置信程度。

In [10]:
evaluation_logits = torch.tensor([[3.0, 1.0, 0.2],  # 样本 1 最看好类别 0
                                  [0.1, 0.5, 2.0],  # 样本 2 最看好类别 2
                                  [0.2, 1.8, 0.4],  # 样本 3 最看好类别 1
                                  [1.2, 0.9, 0.1]])  # 样本 4 最看好类别 0
evaluation_labels = torch.tensor([0, 2, 2, 0])  # 四个样本的真实类别
predictions = evaluation_logits.argmax(dim=1)  # 沿类别维取最大 logit 的索引
correct_mask = predictions == evaluation_labels  # 逐元素比较预测与标签，得到布尔张量
accuracy = correct_mask.float().mean()  # 把 True/False 转成 1/0，再求平均

print("预测类别:", predictions)  # 输出 [0, 2, 1, 0]
print("是否正确:", correct_mask)  # 第三个样本预测错误
print("准确率:", accuracy.item())  # 3 个正确除以 4 个样本，结果为 0.75

预测类别: tensor([0, 2, 1, 0])
是否正确: tensor([ True,  True, False,  True])
准确率: 0.75


## 3.4.9 小结

- 分类模型回答“属于哪一类”，通常同时输出每个类别的概率。
- 独热编码用一个向量表示没有自然顺序的类别。
- softmax 回归使用全连接层产生 logits，再用 softmax 映射为概率分布。
- softmax 不改变 logits 的大小顺序，因此预测时可直接对 logits 使用 `argmax`。
- 小批量矢量化让多个样本同时完成矩阵运算，并沿类别维逐行计算 softmax。
- 最大化标签似然等价于最小化交叉熵损失。
- softmax 交叉熵关于 logits 的梯度为预测概率减真实独热标签：$\hat{\mathbf y}-\mathbf y$。
- 信息论中，交叉熵表示使用预测分布编码真实数据时的预期惊异。
- 准确率等于正确预测数除以样本总数。

## 3.4.10 练习

### 练习 1：softmax、交叉熵与指数族

1. 计算 softmax 交叉熵损失 $l(\mathbf y,\hat{\mathbf y})$ 关于 logits 的二阶导数。
2. 计算 $\operatorname{softmax}(\mathbf o)$ 给出的分类分布协方差，并与二阶导数匹配。

### 练习 2：三个等概率类别的编码

假设三个类别等概率，即概率向量为 $(1/3,1/3,1/3)$。

1. 如果使用固定长度二进制代码，会遇到什么问题？
2. 设计更好的编码。思考联合编码两个独立观测、以及联合编码 $n$ 个观测时会发生什么。

### 练习 3：平滑最大值

定义

$$
\operatorname{RealSoftMax}(a,b)=\log(\exp(a)+\exp(b)).
$$

1. 证明 $\operatorname{RealSoftMax}(a,b)>\max(a,b)$。
2. 证明当 $\lambda>0$ 时，$\lambda^{-1}\operatorname{RealSoftMax}(\lambda a,\lambda b)>\max(a,b)$。
3. 证明 $\lambda\to\infty$ 时，上式趋近于 $\max(a,b)$。
4. 写出对应的 soft-min。
5. 将其扩展到两个以上的数字。

## 3.4.11 练习题参考答案

下面给出推导过程。实际学习时，建议先独立计算，再用答案检查思路。

### 练习 1 答案：softmax、交叉熵与指数族

记 logits 为 $\mathbf o=(o_1,\ldots,o_q)$，softmax 概率为

$$
p_i=\frac{\exp(o_i)}{\sum_k\exp(o_k)}.
$$

对于独热标签 $\mathbf y$，交叉熵可以改写为

$$
l(\mathbf y,\mathbf p)
=-\sum_i y_i\log p_i
=\log\!\left(\sum_k\exp(o_k)\right)-\sum_i y_i o_i,
$$

其中使用了 $\sum_i y_i=1$。对 $o_i$ 求一阶偏导：

$$
\frac{\partial l}{\partial o_i}=p_i-y_i.
$$

再对 $o_j$ 求导。softmax 的偏导为

$$
\frac{\partial p_i}{\partial o_j}
=p_i(\delta_{ij}-p_j),
$$

其中 $\delta_{ij}$ 是克罗内克符号：$i=j$ 时为 1，否则为 0。因此损失函数的 Hessian 元素为

$$
H_{ij}=\frac{\partial^2l}{\partial o_i\partial o_j}
=p_i\delta_{ij}-p_ip_j.
$$

写成矩阵形式：

$$
\boxed{\mathbf H=\operatorname{diag}(\mathbf p)-\mathbf p\mathbf p^\top}.
$$

现在令随机独热向量 $\mathbf Z$ 表示一次类别采样，并满足 $P(Z_i=1)=p_i$。因为一次只能取一个类别，

$$
\mathbb E[\mathbf Z]=\mathbf p,
\qquad
\mathbb E[\mathbf Z\mathbf Z^\top]=\operatorname{diag}(\mathbf p).
$$

所以分类分布的协方差为

$$
\operatorname{Cov}(\mathbf Z)
=\mathbb E[\mathbf Z\mathbf Z^\top]
-\mathbb E[\mathbf Z]\mathbb E[\mathbf Z]^\top
=\operatorname{diag}(\mathbf p)-\mathbf p\mathbf p^\top.
$$

它与交叉熵关于 logits 的 Hessian 完全相同。这也说明 Hessian 是半正定矩阵，交叉熵关于 logits 是凸函数。由于给所有 logits 同时加同一个常数不会改变 softmax，故 $\mathbf H\mathbf 1=0$，Hessian 通常不是正定矩阵。

### 练习 2 答案：三个等概率类别的编码

三个类别等概率时，每个类别的理想信息量为

$$
-\log_2(1/3)=\log_2 3\approx1.585\ \text{比特}.
$$

**1. 固定长度编码的问题**

一个二进制位只能表示 2 种状态，不足以表示 3 类；因此单独编码一个观测至少需要 2 位。例如使用 `00`、`01`、`10`，而 `11` 被浪费。平均码长为 2 比特，高于熵 $\log_2 3$。

**2. 更好的编码**

若逐个编码，可以使用前缀码：

- 类别 A：`0`
- 类别 B：`10`
- 类别 C：`11`

三个类别等概率，所以平均码长为

$$
\frac{1+2+2}{3}=\frac53\approx1.667\ \text{比特/观测},
$$

比固定 2 位更接近理论下界。

联合编码两个独立观测时，共有 $3^2=9$ 个等概率组合。相应 Huffman 前缀码可让 7 个组合使用 3 位、2 个组合使用 4 位，因为

$$
7\cdot2^{-3}+2\cdot2^{-4}=1.
$$

平均总码长为 $29/9$ 比特，因此每个观测平均使用

$$
\frac{29}{18}\approx1.611\ \text{比特},
$$

比逐个编码的 $5/3$ 更短。

联合编码 $n$ 个观测时共有 $3^n$ 个等概率序列。使用固定长度块编码需要

$$
L_n=\left\lceil\log_2(3^n)\right\rceil
=\left\lceil n\log_2 3\right\rceil
$$

比特。于是每个观测的平均码长满足

$$
\log_2 3
\leq\frac{L_n}{n}
<\log_2 3+\frac1n.
$$

当 $n\to\infty$ 时，每个观测的码长趋近于熵 $\log_2 3$。这说明对多个观测进行联合编码，可以把不足一个比特的取整浪费分摊到整个数据块。

### 练习 3 答案：平滑最大值

令 $m=\max(a,b)$。则

$$
\operatorname{RealSoftMax}(a,b)
=\log(\exp(a)+\exp(b))
=m+\log\!\left(\exp(a-m)+\exp(b-m)\right).
$$

括号内一个指数项等于 1，另一个指数项严格大于 0，所以括号内的和严格大于 1。因此

$$
\operatorname{RealSoftMax}(a,b)>m=\max(a,b).
$$

**加入温度参数。** 对 $\lambda>0$，同样提取最大值：

$$
\frac1\lambda\log(\exp(\lambda a)+\exp(\lambda b))
=m+\frac1\lambda\log\!\left(1+\exp(-\lambda|a-b|)\right).
$$

右侧第二项严格大于 0，故整个表达式严格大于 $\max(a,b)$。同时

$$
0<\frac1\lambda\log\!\left(1+\exp(-\lambda|a-b|)\right)
\leq\frac{\log2}{\lambda}.
$$

当 $\lambda\to\infty$ 时，上界趋于 0。根据夹逼定理，

$$
\lim_{\lambda\to\infty}
\frac1\lambda\log(\exp(\lambda a)+\exp(\lambda b))
=\max(a,b).
$$

**对应的 soft-min：** 利用 $\min(a,b)=-\max(-a,-b)$，可定义

$$
\operatorname{SoftMin}_\lambda(a,b)
=-\frac1\lambda\log(\exp(-\lambda a)+\exp(-\lambda b)).
$$

它严格小于 $\min(a,b)$，并在 $\lambda\to\infty$ 时趋近于 $\min(a,b)$。

**推广到 $n$ 个数。** 对 $x_1,\ldots,x_n$，定义

$$
\operatorname{LSE}_\lambda(\mathbf x)
=\frac1\lambda\log\left(\sum_{i=1}^{n}\exp(\lambda x_i)\right).
$$

若 $M=\max_i x_i$，则

$$
M<\operatorname{LSE}_\lambda(\mathbf x)
\leq M+\frac{\log n}{\lambda}
$$

（当只有一个数或允许其他项趋于负无穷时，左侧可取等号）。因此 $\lambda\to\infty$ 时它趋近于最大值。数值计算时应使用稳定形式

$$
\operatorname{LSE}_\lambda(\mathbf x)
=M+\frac1\lambda\log\left(\sum_i\exp(\lambda(x_i-M))\right),
$$

从而避免直接计算很大的指数而溢出。